# 03 WeightWatcher Analysis

Manifest-driven spectral diagnostics and publication exports for canonical scientific runs.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/tmp/wwpgd_v2/data'))
RESULTS_ROOT = Path(os.environ.get('WWGPT_RESULTS_ROOT', os.environ.get('RESULTS_ROOT', '/tmp/wwpgd_v2/real_level0_results_v5/experiments/level_00/multiplier_20')))
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
INCLUDE_LEGACY = False

In [ ]:
import sys
cwd = Path.cwd().resolve()
repo = cwd if (cwd / 'src' / 'wwgpt').exists() else cwd.parent
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))

import pandas as pd
from IPython.display import display
from wwgpt.analysis import build_run_inventory, discover_canonical_runs, load_run_artifacts, resolve_experiment_root
from wwgpt.publication_plots import PublicationPlotConfig, build_all_publication_figures

RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')

## Run and spectral inventory

In [ ]:
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
inventory.to_csv(ANALYSIS_DIR / 'run_inventory.csv', index=False)
display(inventory)
assert runs, f'No canonical scientific runs found under {RESULTS_ROOT}'

In [ ]:
spectral_frames = []
for run in runs:
    artifacts = load_run_artifacts(Path(run['run_dir']))
    spectral = artifacts['spectral'].copy()
    if spectral.empty:
        continue
    spectral_frames.append(spectral.assign(seed=run['seed'], pair_id=run['pair_id'], optimizer_family=run['optimizer_family'], optimizer_raw=run['optimizer_raw']))

spectral_records = pd.concat(spectral_frames, ignore_index=True) if spectral_frames else pd.DataFrame()
spectral_records.to_csv(ANALYSIS_DIR / 'spectral_records_scientific.csv', index=False)
assert not spectral_records.empty, 'Canonical runs contain no spectral records'
assert spectral_records['spectral_estimator'].astype(str).str.lower().eq('weightwatcher').all(), 'Non-WeightWatcher spectral rows found in canonical runs'
display(spectral_records.head())

## Canonical publication figures

All figures use the shared publication API, which exports vector PDF, 300-DPI PNG, exact source data, and metadata with the uncertainty-band definition.

In [ ]:
plot_config = PublicationPlotConfig(band='mean_std', dpi=300)
figure_manifest = build_all_publication_figures(runs, ANALYSIS_DIR, config=plot_config)
figure_manifest = pd.DataFrame(figure_manifest) if not isinstance(figure_manifest, pd.DataFrame) else figure_manifest
figure_manifest.to_csv(ANALYSIS_DIR / 'weightwatcher_plot_manifest.csv', index=False)
display(figure_manifest)